Our benchmark set consists the following state sets:
- random
  - num_qubits
  - num_states
  - seed
- coherent
  - num_states
  - amplitude
  - phase
- entangled
- product states

The POVM will be derived using different QSD methods:

The circuits will be synthesized using different isometry decomposition methods:
csd/ccd (equivalence checking)

The synthesis time is in logger


In [1]:
import sys
sys.path.append("../")

import os
os.environ["IBM_QUANTUM_INSTANCE"] = "crn:v1:bluemix:public:quantum-computing:us-east:a/f071734952cb4c9993a642d0a87d18bb:6501d9e0-9883-41f7-8b19-8fd506d38274::"
os.environ["IBM_QUANTUM_TOKEN"] = "3W50m31-b6uaeYY3fhAK3yQwhzjpU-uY_6-V6-LMDFig"

In [2]:
import logging
logging.basicConfig(
    filename=f"20250912_benchmark_gen.log",
    filemode="a",
    format="{asctime} {levelname} {filename}:{lineno}: {message}",
    datefmt="%Y-%m-%d %H:%M:%S",
    style="{",
    level=logging.INFO,  # Qiskit dumps too many DEBUG messages
    encoding="utf-8",
)

logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('PIL.PngImagePlugin').disabled = True
logging.getLogger('matplotlib.mathtext').disabled = True
logging.getLogger('qiskit.passmanager.base_tasks').disabled = True
logger = logging.getLogger(__name__)

In [3]:
from flow.solve_mix import *
from flow.build_circuits import *
from utils.handy_states import *
from temp.get_random_seeds import *

In [5]:
# def build_benchmark_circuit():
num_qubits = 3
state_dict = coh_symm_small(num_qubits=num_qubits)
num_states = state_dict["num_states"]
qsd_problem = ProblemSpec(
    num_qubits=state_dict["num_qubits"],
    num_states=state_dict["num_states"]
)
qsd_problem.set_states(
    state_type="densitymatrix",
    states=state_dict["dense_mat"],
    overwrite=True,
)

In [6]:
eps = 1e-8
cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": eps}
cvxpy_med_problem = med_problem(qsd_problem)
cvxpy_med_problem.solve(**cvxpy_settings)
vars = cvxpy_med_problem.variables()
med_povm = [var.value for var in vars]

/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18617: UserWarning: Argument sub in putvarboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putvarboundlist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");


In [7]:
med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=0)
med_POVM_qc = POVMCircuit(povm_vectors=med_povm_vectors)
print(len(med_povm_vectors))
print(len(med_POVM_qc.povm_vectors))
med_POVM_qc.num_qubits = num_qubits
med_POVM_qc.num_amps = 2 ** num_qubits
med_POVM_qc.case_id = f"circuits//coherent//coh_q{num_qubits}_n{num_states}_med_fullpovm_csd"
med_qc = med_POVM_qc.build_circuit(scheme="csd")
med_POVM_qc.case_id = f"circuits//coherent//coh_q{num_qubits}_n{num_states}_med_fullpovm_ccd"
med_ccd_qc = med_POVM_qc.build_circuit(scheme="ccd")

24
24


In [8]:
print("MED POVM QC")
print("num_qubits", med_qc.decompose(reps=3).num_qubits)
print("depth", med_qc.decompose(reps=3).depth())
print("gates", med_qc.decompose(reps=3).count_ops())

MED POVM QC
num_qubits 5
depth 701
gates OrderedDict({'u': 632, 'cx': 302})


In [9]:
reduced_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=1e-4)
# reduced_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=5e-1)

reduced_med_POVM_qc = POVMCircuit(povm_vectors=reduced_med_povm_vectors)

print(verify_povm(reduced_med_povm_vectors))
print(len(reduced_med_povm_vectors))
print(len(reduced_med_POVM_qc.povm_vectors))
reduced_med_POVM_qc.num_qubits = num_qubits
reduced_med_POVM_qc.num_amps = 2 ** num_qubits
reduced_med_POVM_qc.case_id = f"circuits//coherent//coh_q{num_qubits}_n{num_states}_med_reducedpovm_csd"
# reduced_med_POVM_qc.fix()
print(len(reduced_med_POVM_qc.povm_vectors))
reduced_med_qc = reduced_med_POVM_qc.build_circuit(scheme="csd")
reduced_med_POVM_qc.case_id = f"circuits//coherent//coh_q{num_qubits}_n{num_states}_med_reducedpovm_ccd"
reduced_med_ccd_qc = reduced_med_POVM_qc.build_circuit(scheme="ccd")

<class 'numpy.ndarray'>
True
18
18
18


In [10]:
print("Reduced MED POVM QC")
print("num_qubits", reduced_med_qc.decompose(reps=3).num_qubits)
print("depth", reduced_med_qc.decompose(reps=3).depth())
print("gates", reduced_med_qc.decompose(reps=3).count_ops())

Reduced MED POVM QC
num_qubits 5
depth 701
gates OrderedDict({'u': 632, 'cx': 302})


In [11]:
circuit = qiskit.qasm2.load(
    "circuits/coherent/coh_q3_n3_med_fullpovm_ccd_no_backend.qasm",
    custom_instructions=qiskit.qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)